In [11]:
import csv

import os
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
root_override = os.environ.get("OCL_PROJECT_ROOT")
candidates = [Path(root_override).resolve()] if root_override else [cwd, *cwd.parents]
project_root = next((p for p in candidates if (p / "src/dataset.py").is_file()), None)
if project_root is None:
    raise RuntimeError("Open the notebook from the OCL workspace or set OCL_PROJECT_ROOT.")

sys.path.insert(0, str(project_root))
data_root = Path(os.environ.get("OCL_DATA_ROOT", project_root / "data"))
resource_root = Path(os.environ.get("OCL_RESOURCE_ROOT", data_root / "resources"))

csv_path = data_root / "final_audit" / "valid_instances.csv"
with csv_path.open("r", encoding="utf-8-sig", newline="") as stream:
    valid_instances = list(csv.DictReader(stream))

print("Number of instances:", len(valid_instances))
print("Fields:", valid_instances[0].keys())
print("First instance:", valid_instances[0])

Number of instances: 11231
Fields: dict_keys(['source', 'path', 'object_index', 'category', 'width', 'height', 'sha256', 'raw_xyxy', 'mapped_xyxy', 'crop_xyxy', 'clipped', 'attribute_ids', 'affordance_ids'])
First instance: {'source': 'ImageNet', 'path': 'ImageNet/train/n02085620/n02085620_13855.JPEG', 'object_index': '0', 'category': 'dog', 'width': '375', 'height': '500', 'sha256': 'a9a2d5fef1d61f5c7a78c1f9b365782b1032a7343ed10c7665d944dfdccff34f', 'raw_xyxy': '[11, 52, 198, 231]', 'mapped_xyxy': '[16.11328125, 101.5625, 290.0390625, 451.171875]', 'crop_xyxy': '[16, 101, 291, 452]', 'clipped': 'False', 'attribute_ids': '[4, 8, 16, 17, 20, 21, 25, 31, 35, 37, 38, 39, 50, 52, 53, 58, 65, 66, 76, 92]', 'affordance_ids': '[9, 10, 11, 14, 16, 24, 31, 33, 36, 37, 40, 41, 44, 49, 50, 52, 53, 54, 58, 67, 71, 72, 73, 74, 75, 76, 83, 84, 98, 101, 102, 104, 105, 107, 108, 110, 111, 112, 117, 121, 124, 130, 133, 135, 153, 155, 156, 162, 163, 164, 165, 168]'}


In [12]:
from src.dataset import OCLDataset

In [4]:
from src.model import load_model

model, preprocess, _ = load_model("ViT-B/32", "cpu", os.environ.get("OCL_WEIGHTS_DIR"))

d:\miniconda3\envs\ocl\Lib\site-packages\torch\jit\_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(


In [5]:
dataset = OCLDataset(
    data_root=data_root,
    resource_root=resource_root,
    preprocess=preprocess
)

print(len(dataset))

11231


In [6]:
sample = dataset[0]

for key, value in sample.items():
    print(key, value.shape, value.dtype)

image torch.Size([3, 224, 224]) torch.float32
category torch.Size([]) torch.int64
attribute torch.Size([114]) torch.float32
affordance torch.Size([170]) torch.float32


In [8]:
import torch

print(dataset.categories[sample["category"].item()])

print(sample["attribute"].sum())
print(sample["affordance"].sum())

print(torch.where(sample["attribute"] == 1)[0])

dog
tensor(20.)
tensor(52.)
tensor([ 4,  8, 16, 17, 20, 21, 25, 31, 35, 37, 38, 39, 50, 52, 53, 58, 65, 66,
        76, 92])


In [9]:
from torch.utils.data import DataLoader

test_loader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0,
    drop_last=False
)

batch = next(iter(test_loader))

for key, value in batch.items():
    print(key, value.shape)

image torch.Size([4, 3, 224, 224])
category torch.Size([4])
attribute torch.Size([4, 114])
affordance torch.Size([4, 170])


In [10]:
print(len(dataset))
print(len(test_loader))

last_batch = next(iter(DataLoader(
    dataset,
    batch_size=4,
    sampler=range(len(dataset) - 3, len(dataset))
)))

for key, value in last_batch.items():
    print(key, value.shape)

11231
2808
image torch.Size([3, 3, 224, 224])
category torch.Size([3])
attribute torch.Size([3, 114])
affordance torch.Size([3, 170])
